In [1]:
# validate the downscaled data using the same framework with all raw datasets

# libraries
import pandas as pd
import numpy as np
import glob
import os
import rasterio
import HydroErr as he
import concurrent.futures
from pyproj import Transformer

#paths
insitu_tmax_raw = pd.read_csv(r"..\Data\Processed\Insitu\insitu_maxtemp_1991_2020.csv")
insitu_tmin_raw = pd.read_csv(r"..\Data\Processed\Insitu\insitu_mintemp_1991_2020.csv")
tmax_train_df = fr"..\Data_Temp\Processed\Split_stations\train_station_tmax.csv"
tmax_test_df = fr"..\Data_Temp\Processed\Split_stations\test_station_tmax.csv"
tmin_train_df = fr"..\Data_Temp\Processed\Split_stations\train_station_tmin.csv"
tmin_test_df = fr"..\Data_Temp\Processed\Split_stations\test_station_tmin.csv"
output_dir = fr"..\Data_Temp\Processed\Monthly_downscaling\Validation_results"

#parameters
datasets = ['ERA5L', 'HAR', 'CHIRTS', 'IS_K', 'DS']
start_year, start_month = 2001, 1
end_year,   end_month   = 2005, 12
start_date = pd.Timestamp(f"{start_year}-{start_month:02d}-01")
end_date   = pd.Timestamp(f"{end_year}-{end_month:02d}-01") + pd.offsets.MonthEnd(0)


In [2]:
# 2 Preprocess insitu monthly 

print("==========Aggregating insitu tmax to monthly totals==============")
insitu_tmax_raw["date"] = pd.to_datetime(insitu_tmax_raw[['year', 'month', 'day']])
insitu_tmax_raw = insitu_tmax_raw.rename(columns = {'lat': 'y', 'long': 'x',
                                                            'max_temp': 'IS_tmax'})
insitu_tmax_raw = insitu_tmax_raw[insitu_tmax_raw["IS_tmax"] > -9999]

insitu_tmax_monthly = (
    insitu_tmax_raw
    .groupby(['index_no', 'y', "x", 'year', 'month'])
    .agg(
        IS_tmax_average = ('IS_tmax', 'mean'),
        valid_days = ('IS_tmax', 'count')
    )
    .reset_index()
)

insitu_tmax_monthly = insitu_tmax_monthly[
    (insitu_tmax_monthly["year"]    >=  start_year) &
    (insitu_tmax_monthly["year"]    <=  end_year)
].copy()

insitu_tmax_monthly[["x", "y"]] = insitu_tmax_monthly[["x", "y"]].round(5)

train_ids_tmax = set(pd.read_csv(tmax_train_df)["index_no"])
test_ids_tmax = set(pd.read_csv(tmax_test_df)["index_no"])

insitu_tmax_monthly_train = insitu_tmax_monthly[insitu_tmax_monthly["index_no"].isin(train_ids_tmax)].copy()
insitu_tmax_monthly_test = insitu_tmax_monthly[insitu_tmax_monthly["index_no"].isin(test_ids_tmax)].copy()

# Unique test stations — order must stay fixed (iloc used in sampling)
unique_stn_tmax = (
    insitu_tmax_monthly_test[['index_no', 'x', 'y']]
    .drop_duplicates('index_no')
    .reset_index(drop=True)
)
stn_points_tmax = list(zip(unique_stn_tmax['x'], unique_stn_tmax['y']))  # (lon, lat)


print("\n==========Aggregating insitu tmin to monthly totals==============")
insitu_tmin_raw["date"] = pd.to_datetime(insitu_tmin_raw[['year', 'month', 'day']])
insitu_tmin_raw = insitu_tmin_raw.rename(columns = {'lat': 'y', 'long': 'x',
                                                            'min_temp': 'IS_tmin'})
insitu_tmin_raw = insitu_tmin_raw[insitu_tmin_raw["IS_tmin"] >= -9999]

insitu_tmin_monthly = (
    insitu_tmin_raw
    .groupby(['index_no', 'y', "x", 'year', 'month'])
    .agg(
        IS_tmin_average = ('IS_tmin', 'mean'),
        valid_days = ('IS_tmin', 'count')
    )
    .reset_index()
)


insitu_tmin_monthly = insitu_tmin_monthly[
    (insitu_tmin_monthly["year"]    >=  start_year) &
    (insitu_tmin_monthly["year"]    <=  end_year)
].copy()

insitu_tmin_monthly[["x", "y"]] = insitu_tmin_monthly[["x", "y"]].round(5)

# Train/test split
train_ids_tmin = set(pd.read_csv(tmin_train_df)["index_no"])
test_ids_tmin = set(pd.read_csv(tmin_test_df)["index_no"])

insitu_tmin_monthly_train = insitu_tmin_monthly[insitu_tmin_monthly["index_no"].isin(train_ids_tmin)].copy()
insitu_tmin_monthly_test = insitu_tmin_monthly[insitu_tmin_monthly["index_no"].isin(test_ids_tmin)].copy()


# Unique test stations — order must stay fixed (iloc used in sampling)
unique_stn_tmin = (
    insitu_tmin_monthly_test[['index_no', 'x', 'y']]
    .drop_duplicates('index_no')
    .reset_index(drop=True)
)
stn_points_tmin = list(zip(unique_stn_tmin['x'], unique_stn_tmin['y']))  # (lon, lat)

==========Aggregating insitu tmax to monthly totals==============

==========Aggregating insitu tmin to monthly totals==============


In [ ]:
#Helper

def extract_date(filepath, dataset):
    """Parse date from filename for both daily and monthly naming conventions."""
    fname = os.path.basename(filepath)
    return pd.to_datetime(fname[-10:-4], format='%Y%m')


def get_dataset_paths(t_variable):
    return {'ERA5L': fr"..\Data_Temp\Processed\Monthly_downscaling\Input_rasters\era5l",
    'HAR': fr"..\Data_Temp\Processed\Monthly_downscaling\Input_rasters\har",
    'CHIRTS':fr"..\Data_Temp\Processed\Monthly_downscaling\Input_rasters\chirts",
    'IS_K': fr"..\Data_Temp\Datasets\Kriging_monthly\{t_variable}",
    'DS': fr"..\Data_Temp\Processed\Monthly_downscaling\Final_products\{t_variable}"
    }

def build_dataset_results(dataset_paths, datasets, t_variable):
    """Build {dataset}_files dict pre-filtered to analysis period."""
    dataset_results = {}
    for dataset in datasets:
        if dataset in ['ERA5L', 'HAR', 'CHIRTS']:
            files = sorted(glob.glob(os.path.join(dataset_paths[dataset], f"{dataset.lower()}_{t_variable}_average_*.tif")))
        else:
            files = sorted(glob.glob(os.path.join(dataset_paths[dataset], "*.tif")))
        filtered = []
        for f in files:
            try:
                d = extract_date(f, dataset)
                if start_date <= d <= end_date:
                    filtered.append(f)
            except Exception as e:
                print(f"  Skipping {os.path.basename(f)}: {e}")
        dataset_results[f'{dataset}_files'] = filtered
        print(f"{dataset}: {len(filtered)} files in period")
    return dataset_results

In [4]:
def _sample_file(file, dataset, stn_points, unique_stn, t_variable):
    """Sample all station points from a single raster file."""
    date_obj = extract_date(file, dataset)
    try:
        with rasterio.open(file) as src:
            if src.crs.to_epsg() != 4326:
                transformer = Transformer.from_crs(
                    "EPSG:4326", src.crs, always_xy=True
                )
                pts = [transformer.transform(x, y) for x, y in stn_points]
            else:
                pts = stn_points

            sampled = list(src.sample(pts))

        return [
            {
                'date':              date_obj,
                'stn_id':            unique_stn.iloc[i]['index_no'],
                f'{dataset}_{t_variable}':   v[0],
            }
            for i, v in enumerate(sampled)
        ]
    except Exception as e:
        print(f"  Error processing {os.path.basename(file)}: {e}")
        return []


def extract_monthly_datasets(dataset_results, stn_points, unique_stn, datasets, t_variable):
    """
    For already-monthly datasets
      Sample each file concurrently and merge at monthly level.
    """
    df_final = None

    for dataset in datasets:
        files = dataset_results.get(f'{dataset}_files', [])
        if not files:
            print(f"Warning: no files found for {dataset}, skipping.")
            continue
        print(f"Processing {dataset} ({len(files)} monthly files)...")

        records = []
        with concurrent.futures.ThreadPoolExecutor() as executor:
            for result in executor.map(
                lambda f: _sample_file(f, dataset, stn_points, unique_stn, t_variable), files
            ):
                records.extend(result)

        df_dataset = pd.DataFrame(records)
        if df_dataset.empty:
            print(f"  Warning: no data extracted for {dataset}")
            continue

        df_dataset['date'] = pd.to_datetime(df_dataset['date'])
        df_dataset = df_dataset.assign(
            year  = lambda d: d['date'].dt.year,
            month = lambda d: d['date'].dt.month
        ).drop(columns='date')

        df_final = (
            df_dataset if df_final is None
            else pd.merge(df_final, df_dataset,
                          on=['stn_id', 'year', 'month'], how='outer')
        )
        print(f"  → merged shape: {df_final.shape}")

    return df_final

In [6]:
# Build dataset results - Tmax
tmax_results = build_dataset_results(get_dataset_paths("tmax"), datasets, "tmax")

df_grid_tmax = extract_monthly_datasets(
    tmax_results, stn_points_tmax, unique_stn_tmax, datasets, "tmax" 
)

df_validation_tmax = pd.merge(
    insitu_tmax_monthly_test.rename(columns={'index_no': 'stn_id'}),
    df_grid_tmax,
    on=['stn_id', 'year', 'month'],
    how='inner'
)
df_validation_tmax.to_csv(os.path.join(output_dir, "Tmax_validation_df.csv"), index = False)

# Build dataset results - Tmin
tmin_results = build_dataset_results(get_dataset_paths("tmin"), datasets, "tmin")

df_grid_tmin = extract_monthly_datasets(
    tmin_results, stn_points_tmin, unique_stn_tmin, datasets, "tmin" 
)

df_validation_tmin = pd.merge(
    insitu_tmin_monthly_test.rename(columns={'index_no': 'stn_id'}),
    df_grid_tmin,
    on=['stn_id', 'year', 'month'],
    how='inner'
)
df_validation_tmin.to_csv(os.path.join(output_dir, "Tmin_validation_df.csv"), index = False)

ERA5L: 60 files in period
HAR: 60 files in period
CHIRTS: 60 files in period
IS_K: 60 files in period
DS: 60 files in period
Processing ERA5L (60 monthly files)...
  → merged shape: (1380, 4)
Processing HAR (60 monthly files)...
  → merged shape: (1380, 5)
Processing CHIRTS (60 monthly files)...
  → merged shape: (1380, 6)
Processing IS_K (60 monthly files)...
  → merged shape: (1380, 7)
Processing DS (60 monthly files)...
  → merged shape: (1380, 8)
ERA5L: 60 files in period
HAR: 60 files in period
CHIRTS: 60 files in period
IS_K: 60 files in period
DS: 60 files in period
Processing ERA5L (60 monthly files)...
  → merged shape: (1380, 4)
Processing HAR (60 monthly files)...
  → merged shape: (1380, 5)
Processing CHIRTS (60 monthly files)...
  → merged shape: (1380, 6)
Processing IS_K (60 monthly files)...
  → merged shape: (1380, 7)
Processing DS (60 monthly files)...
  → merged shape: (1380, 8)


In [ ]:
#column normalization function
def normalize_column(col):
    return (col - col.min()) / (col.max() - col.min())
          
#calculate accuracy of datasets
def calculate_accuracy(dataframe_temp, t_variable, datasets_list):
    df_accuracy = pd.DataFrame()

    for dataset in datasets_list:
        dataset_accuracy = {'dataset': dataset}

        # Dynamically reference temperature columns
        grid_col = f'{dataset}_{t_variable}'
        insitu_col = f'IS_{t_variable}_average'
        
        # Check if dataset was successfully extracted
        if grid_col not in dataframe_temp.columns:
            print(f"Skipping {dataset} - column not found.")
            continue

        temp_grid_array = dataframe_temp[grid_col].values
        temp_insitu_array = dataframe_temp[insitu_col].values
    
        # Drop null values
        valid_indices = np.logical_and.reduce([
            ~np.isnan(temp_grid_array),
            ~np.isnan(temp_insitu_array)
        ])
    
        observations = np.sum(valid_indices)
        dataset_accuracy['observations_n'] = observations

        if observations == 0:
            continue

        temp_grid_array = temp_grid_array[valid_indices]
        temp_insitu_array = temp_insitu_array[valid_indices]

        # Calculate metrics
        dataset_accuracy['correlation'] = he.spearman_r(temp_grid_array, temp_insitu_array)
        dataset_accuracy['nrmse'] = he.nrmse_mean(temp_grid_array, temp_insitu_array)
        dataset_accuracy['pbias'] = 100 * np.sum(temp_grid_array - temp_insitu_array) / np.sum(temp_insitu_array)
        dataset_accuracy['rsd'] = (np.std(temp_grid_array)) / (np.std(temp_insitu_array))

        df_dataset_accuracy = pd.DataFrame([dataset_accuracy])
        df_accuracy = pd.concat([df_accuracy, df_dataset_accuracy], ignore_index=True)

    # Normalization
    df_accuracy['abs_pbias'] = df_accuracy['pbias'].abs()
    df_accuracy['diff_rsd'] = (df_accuracy['rsd'] - 1).abs()

    df_accuracy_normalized = df_accuracy.copy().drop(columns=['pbias'])

    metrics = ['correlation', 'nrmse', 'abs_pbias', 'diff_rsd']
    for column in metrics:
        df_accuracy_normalized[column] = normalize_column(df_accuracy_normalized[column])

    # Invert correlation (1 = ideal, 0 = worst)
    df_accuracy_normalized['correlation'] = 1 - df_accuracy_normalized['correlation']

    # Calculate CPI
    df_accuracy_normalized['cpi'] = (df_accuracy_normalized['correlation'] + 
                                     df_accuracy_normalized['nrmse'] + 
                                     df_accuracy_normalized['abs_pbias'] + 
                                     df_accuracy_normalized['diff_rsd'])
    df_accuracy_normalized['cpi_rank'] = df_accuracy_normalized['cpi'].rank(ascending=True)

    df_accuracy_final = df_accuracy_normalized[['dataset','cpi','cpi_rank']].merge(
        df_accuracy[['dataset','correlation','nrmse','pbias','rsd', 'observations_n']], 
        on=['dataset'], 
        how='inner'
    )
    return df_accuracy_final

In [8]:
results_tmax = calculate_accuracy(df_validation_tmax, "tmax", datasets)
results_tmax

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n
0,ERA5L,3.512644,5.0,0.808021,0.194598,-14.955805,0.942227,1333
1,HAR,2.678715,4.0,0.812067,0.176655,-12.574310,1.004801,1333
2,CHIRTS,0.956614,2.0,0.929215,0.078684,-0.273597,0.934870,1333
3,IS_K,1.657850,3.0,0.906972,0.089994,1.163129,0.891868,1333
4,DS,0.813109,1.0,0.966566,0.060253,2.368114,0.925920,1333


In [9]:
results_tmin = calculate_accuracy(df_validation_tmin, "tmin", datasets)
results_tmin

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n
0,ERA5L,1.618827,4.0,0.873051,0.241609,-8.771782,0.927510,1332
1,HAR,0.523668,2.0,0.906173,0.186569,1.874051,0.939647,1332
2,CHIRTS,1.155042,3.0,0.941494,0.183254,9.381035,0.997684,1332
3,IS_K,3.729860,5.0,0.749714,0.692792,7.104575,1.707043,1332
4,DS,0.135630,1.0,0.956097,0.136354,0.954072,0.902101,1332
